### <font color='teal'> ***Importing Libraries*** </font>

In [19]:
from bs4 import BeautifulSoup
import requests
import json
import random
from crewai import Agent
from crewai_tools import BaseTool, SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict, Type

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

### <font color='teal'> ***Urls to extract data from*** </font>

In [11]:
URLS = [
        "https://www.mylondon.news/all-about/crime",
#         "https://www.birminghammail.co.uk/all-about/crime",
#         "https://www.manchestereveningnews.co.uk/all-about/crime",
#         "https://www.liverpoolecho.co.uk/all-about/crime",
#         "https://www.walesonline.co.uk/all-about/crime",
    ]

### <font color='teal'> ***Getting News Links*** </font>

In [12]:
urls = []
links = []

link_class = ['headline']

for URL in URLS:
    url = URL
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'html')
    
    for class_name in link_class:
        sublist = [a.get('href') for a in soup.find_all('a', class_=class_name)]
    
    # Select 5 random elements from the sublist
    random_subset = random.sample(sublist, min(5, len(sublist)))
    
    urls.append(random_subset)

# Flatten the list of lists into a single list
links = [link for sublist in urls for link in sublist]

### <font color='teal'> ***Setting the Classes for different elements*** </font>

In [13]:
class_names = {
    'title': [
        'section-theme-background-indicator publication-font',
        'HeadingOne_heading-one__mmNWJ __className_9cd0cd'
    ],
    'subtitle': [
        'sub-title',
        'LeadText_lead-text__wd_PA'
    ],
    'paragraphs': [
        'article-body',
        'Paragraph_paragraph-text__PVKlh'
    ]
}

### <font color='teal'> ***Extracting the necessary data*** </font>

In [14]:
data = []

for link in links:
    try:
        article_1 = requests.get(link)
        article = BeautifulSoup(article_1.text, 'html.parser')

        # Extract title
        title = None
        for class_name in class_names['title']:
            title = article.find('h1', class_=class_name)
            if title:
                title = title.text.strip()
                break

        # Extract subtitle
        subtitle = None
        for class_name in class_names['subtitle']:
            subtitle = article.find('h2', class_=class_name)
            if subtitle:
                subtitle = subtitle.text.strip()
                break

        # Extract paragraphs
        paragraphs = []
        for class_name in class_names['paragraphs']:
            paragraphs.extend(article.find_all(['p', 'div'], class_=class_name))

        main_paragraphs = [p.text.strip() for p in paragraphs]

        article_data = {
            'link': link,
            'title': title,
            'subtitle': subtitle,
            'paragraphs': main_paragraphs
        }

        data.append(article_data)

    except Exception as e:
        print(f"Error occurred while scraping {link}: {str(e)}")

### <font color='teal'> ***Storing the data in json format*** </font>

In [15]:
# Save the data to a JSON file
with open('scraped_data.json', 'w') as f:
    json.dump(data, f, indent=4)
    print("Data saved to scraped_data.json")

Data saved to scraped_data.json


### <font color='orange'> ***CrewAi Custom Tool***

In [18]:
# Define the input schema for the tool
class ScrapingToolInput(BaseModel):
    urls: List[str] = Field(
        ..., 
        description="A list of URLs to scrape data from."
    )
    num_links: int = Field(
        5, 
        description="The number of random links to extract from each URL."
    )

# Define the output schema for the tool
class ScrapingToolOutput(BaseModel):
    data: List[Dict] = Field(
        ..., 
        description="A list of dictionaries containing scraped data with title, subtitle, paragraphs, and link."
    )

# Define the custom scraping tool
class WebScrapingTool(BaseTool):
    name: str = "Web Scraping Tool"
    description: str = (
        "Scrapes crime-related news articles from provided URLs. Extracts titles, subtitles, and paragraphs."
    )
    args_schema: Type[BaseModel] = ScrapingToolInput
    return_schema: Type[BaseModel] = ScrapingToolOutput

    def _run(self, urls: List[str], num_links: int = 5) -> ScrapingToolOutput:
        # Define classes for elements to scrape
        link_class = ['headline']
        class_names = {
            'title': [
                'section-theme-background-indicator publication-font',
                'HeadingOne_heading-one__mmNWJ __className_9cd0cd'
            ],
            'subtitle': [
                'sub-title',
                'LeadText_lead-text__wd_PA'
            ],
            'paragraphs': [
                'article-body',
                'Paragraph_paragraph-text__PVKlh'
            ]
        }

        links = []
        # Scrape links from provided URLs
        for url in urls:
            try:
                page = requests.get(url)
                soup = BeautifulSoup(page.text, 'html.parser')
                sublist = [a.get('href') for a in soup.find_all('a', class_=link_class[0])]
                # Select random links
                random_subset = random.sample(sublist, min(num_links, len(sublist)))
                links.extend(random_subset)
            except Exception as e:
                print(f"Error occurred while fetching links from {url}: {str(e)}")
        
        data = []
        # Scrape data from links
        for link in links:
            try:
                article_1 = requests.get(link)
                article = BeautifulSoup(article_1.text, 'html.parser')

                # Extract title
                title = None
                for class_name in class_names['title']:
                    title = article.find('h1', class_=class_name)
                    if title:
                        title = title.text.strip()
                        break

                # Extract subtitle
                subtitle = None
                for class_name in class_names['subtitle']:
                    subtitle = article.find('h2', class_=class_name)
                    if subtitle:
                        subtitle = subtitle.text.strip()
                        break

                # Extract paragraphs
                paragraphs = []
                for class_name in class_names['paragraphs']:
                    paragraphs.extend(article.find_all(['p', 'div'], class_=class_name))

                main_paragraphs = [p.text.strip() for p in paragraphs]

                article_data = {
                    'link': link,
                    'title': title,
                    'subtitle': subtitle,
                    'paragraphs': main_paragraphs
                }
                data.append(article_data)

            except Exception as e:
                print(f"Error occurred while scraping {link}: {str(e)}")

        # Return the scraped data
        return ScrapingToolOutput(data=data)

In [19]:
scraping_tool = WebScrapingTool()

In [28]:
# Run the tool
result = scraping_tool._run(
    urls=[
        "https://www.mylondon.news/all-about/crime",
        "https://www.birminghammail.co.uk/all-about/crime",
        "https://www.manchestereveningnews.co.uk/all-about/crime",
        "https://www.liverpoolecho.co.uk/all-about/crime",
        "https://www.walesonline.co.uk/all-about/crime"
    ],
    num_links=7
)

In [30]:
# Save the result to JSON
with open('crew_scraped_data.json', 'w') as f:
    json.dump(result.dict(), f, indent=4)
    print("Data saved to crew_scraped_data.json")

Data saved to crew_scraped_data.json


### <font color='sky blue'> ***Generating PDF***

In [35]:
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, PageBreak
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.colors import blue
from reportlab.lib.units import inch
import json
from datetime import date

In [38]:
# Register fonts (update with your font paths if needed)
pdfmetrics.registerFont(TTFont("Times-Roman", "fonts\\timr45w.ttf"))
pdfmetrics.registerFont(TTFont("Times-Bold", "fonts\\times-bold.ttf"))
pdfmetrics.registerFont(TTFont("Times-Italic", "fonts\\times-italic.ttf"))

def generate_pdf_from_json(input_json_path, output_pdf_path):
    # Read JSON data
    with open(input_json_path, "r", encoding="utf-8") as file:
        data = json.load(file)["data"]

    # Setup the PDF document
    doc = SimpleDocTemplate(output_pdf_path, pagesize=letter)
    story = []

    # Define styles
    title_style = ParagraphStyle(
        name="TitleStyle",
        fontName="Times-Bold",
        fontSize=24,
        alignment=TA_CENTER,
        spaceAfter=10
    )
    subtitle_style = ParagraphStyle(
        name="SubtitleStyle",
        fontName="Times-Italic",
        fontSize=16,
        alignment=TA_CENTER,
        spaceAfter=20,
        textColor=blue  # Change subtitle color to blue
    )
    headline_style = ParagraphStyle(
        name="HeadlineStyle",
        fontName="Times-Bold",
        fontSize=16,
        alignment=TA_JUSTIFY,
        spaceAfter=12
    )
    content_style = ParagraphStyle(
        name="ContentStyle",
        fontName="Times-Italic",  # Make content italic
        fontSize=10,  # Reduce the font size
        alignment=TA_JUSTIFY,
        spaceAfter=10
    )
    link_style = ParagraphStyle(
        name="LinkStyle",
        fontName="Times-Roman",
        fontSize=12,
        alignment=TA_JUSTIFY,
        textColor=blue,
        underline=True,
        spaceAfter=10
    )

    # Add title page
    today = date.today().strftime("%B %d, %Y")
    story.append(Paragraph("UK Crime Chronicles", title_style))
    story.append(Paragraph("A Deep Dive into recent Headlines and Stories", subtitle_style))
    story.append(Paragraph(f"{today}", subtitle_style))
    story.append(PageBreak())

    # Add content from JSON
    for article in data:
        # Add title
        story.append(Paragraph(article["title"], headline_style))

        # Add paragraphs (content style with italics and smaller font)
        for paragraph in article["paragraphs"]:
            story.append(Paragraph(paragraph, content_style))

        # Add source (clickable link without showing the URL)
        if "link" in article and article["link"]:
            story.append(Paragraph(f"<a href='{article['link']}'>Source</a>", link_style))

        story.append(Spacer(1, 0.5 * inch))  # Add spacing between articles
        story.append(PageBreak())

    # Footer with page number
    def add_page_footer(canvas, doc):
        canvas.saveState()
        canvas.setFont("Times-Roman", 10)
        canvas.drawCentredString(letter[0] / 2, 0.5 * inch, str(doc.page))
        canvas.restoreState()

    # Build the PDF
    doc.build(story, onFirstPage=add_page_footer, onLaterPages=add_page_footer)
    print(f"PDF report generated and saved in: {output_pdf_path}")


In [39]:
generate_pdf_from_json("crew_scraped_data.json", "news_report.pdf")

PDF report generated and saved in: news_report.pdf
